In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# แสดงภาพ circuit timing

<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
แม้ว่า [timeline drawer](/guides/visualize-circuit-timing) ที่ built in ใน Qiskit จะมีประโยชน์สำหรับ static circuits แต่อาจไม่สะท้อน timing ที่แม่นยำของ [dynamic circuits](/guides/classical-feedforward-and-control-flow) เนื่องจาก implicit operations เช่น broadcasting และ branch determination ในส่วนของการรองรับ dynamic circuits Qiskit Runtime ส่งคืนข้อมูล circuit timing ที่แม่นยำภายในผลลัพธ์ job เมื่อมีการร้องขอ

> **Note:** - นี่เป็นฟังก์ชันทดลอง อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง
> - ฟังก์ชันนี้ใช้ได้เฉพาะกับ Qiskit Runtime Sampler jobs เท่านั้น
> - แม้ว่าเวลา circuit ทั้งหมดจะถูกส่งคืนใน "compilation" metadata แต่นี่ไม่ใช่เวลาที่ใช้สำหรับการเรียกเก็บเงิน (QPU time)
### เปิดใช้งานการดึงข้อมูล timing
เพื่อเปิดใช้งานการดึงข้อมูล timing ให้ตั้งค่า experimental flag `scheduler_timing` เป็น `True` เมื่อรัน primitive job

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### เข้าถึงข้อมูล circuit timing
เมื่อมีการร้องขอ ข้อมูล circuit timing สำหรับแต่ละ PUB จะถูกส่งคืนใน job result metadata ภายใต้ `["compilation"]["scheduler_timing"]["timing"]` field นี้มีข้อมูล timing ดิบ เพื่อแสดงข้อมูล timing ให้ใช้เครื่องมือแสดงภาพในตัวตามที่อธิบายในส่วน [แสดงภาพ timings](#visualize-timings)

ใช้โค้ดต่อไปนี้เพื่อเข้าถึงข้อมูล circuit timing สำหรับ PUB แรก:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### ทำความเข้าใจข้อมูล timing ดิบ
แม้ว่าการแสดงภาพข้อมูล circuit timing โดยใช้เมธอด `draw_circuit_schedule_timing` จะเป็นกรณีการใช้งานที่พบบ่อยที่สุด แต่อาจเป็นประโยชน์ที่จะเข้าใจโครงสร้างของข้อมูล timing ดิบที่ส่งคืน ซึ่งอาจช่วยให้คุณดึงข้อมูลแบบ programmatic ได้

ข้อมูล timing ที่ส่งคืนใน `["compilation"]["scheduler_timing"]["timing"]` เป็น list ของ strings โดยแต่ละ string แสดงคำสั่งเดียวบน channel หนึ่ง และคั่นด้วยเครื่องหมายจุลภาคเป็นประเภทข้อมูลต่อไปนี้:

- `Branch` - กำหนดว่าคำสั่งอยู่ใน control flow (then / else) หรือ main branch
- `Instruction` - Gate และ qubit ที่จะดำเนินการ
- `Channel` - Channel ที่ถูก assign ด้วยคำสั่ง สามารถเป็นหนึ่งในต่อไปนี้:
      - `Qubit x` - Drive channel สำหรับ qubit _x_
      - `AWGRx_y` (arbitrary waveform generator readout) - ใช้โดย readout channels เพื่อสื่อสารเมื่อวัด qubits อาร์กิวเมนต์ _x_ และ _y_ สอดคล้องกับ readout instrument ID และหมายเลข qubit ตามลำดับ
- `T0` - เวลาเริ่มต้นของคำสั่งภายใน complete schedule
- `Duration` - ระยะเวลาของคำสั่ง ในหน่วยของ _dt_ วินาที โดย 1 dt = 1 scheduling cycle คุณสามารถหาค่า `dt` ของ backend ได้โดยใช้ [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)
- `Pulse` - ประเภทของการดำเนินการ pulse ที่ใช้

ตัวอย่าง:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### แสดงภาพ timings
ด้วย `qiskit-ibm-runtime` v0.43.0 หรือใหม่กว่า คุณสามารถแสดงภาพ circuit timings ได้ เพื่อแสดงภาพ timings คุณต้องแปลง result metadata เป็น `fig` ก่อนโดยใช้ [เมธอด `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) เมธอดนี้ส่งคืน `plotly` figure ที่คุณสามารถแสดงโดยตรง บันทึกเป็นไฟล์ หรือทั้งสองอย่าง สำหรับข้อมูลเพิ่มเติมเกี่ยวกับคำสั่ง `plotly` ที่จะใช้ ดู [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) และ [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Hovering over the output shows information such as the start, finish, and duration.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Example of a generated figure')
#### ทำความเข้าใจรูปภาพที่สร้างขึ้น
ภาพของ circuit timing data ที่ออกมาจาก `draw_circuit_schedule_timing` สื่อข้อมูลต่อไปนี้:

- แกน X คือเวลาในหน่วยของ _dt_ วินาที โดย 1 dt = 1 scheduling cycle คุณสามารถหาค่า `dt` ของ backend ได้โดยใช้ [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)
- แกน Y คือ channel (คิดว่า channels เป็น instruments ที่ส่ง pulses)
    - `Receive channel` - Channel เดียวที่ไม่ใช่ instrument ด้วยตัวเอง เป็นคำสั่งที่รันบน channels ทั้งหมดที่เป็นส่วนหนึ่งของกระบวนการสื่อสารกับ hub ในเวลานั้น
    - `Qubit x` - Drive channel สำหรับ qubit x
    - `AWGRx_y` (arbitrary waveform generator readout) - ใช้โดย readout channels เพื่อสื่อสารเมื่อวัด qubits อาร์กิวเมนต์ _x_ และ _y_ สอดคล้องกับ readout instrument ID และหมายเลข qubit ตามลำดับ
    - `Hub` - ควบคุม broadcasting

นอกจากนี้ แต่ละคำสั่งมีรูปแบบ *X_Y* โดย *X* คือชื่อของคำสั่งและ *Y* คือประเภท pulse ประเภท `play` ใช้ control pulses และ `capture` บันทึก state ของ qubit คุณยังสามารถ hover เหนือแต่ละคำสั่งเพื่อดูรายละเอียดเพิ่มเติม ตัวอย่างเช่น รูปก่อนหน้าแสดง control pulse สำหรับ X gate ที่ใช้กับ qubit 10 ที่ 1161 dt
### ตัวอย่าง end-to-end
ตัวอย่างนี้แสดงวิธีเปิดใช้งาน option รับข้อมูล circuit timing จาก metadata และแสดงเป็นภาพ

ขั้นแรก ตั้งค่า environment กำหนด circuits และแปลงเป็น ISA circuits และกำหนดและรัน jobs

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

จากนั้น รับ circuit schedule timing: